# Google Store Sales Analysis

**Dataset:** Google Store e-commerce orders (2017) — 18,782 records  
**Author:** Douglas Piangers Mengue  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn

---

## 1. Project Introduction

This project analyzes order data from the Google Store to understand sales performance across regions, product categories, and acquisition channels.

**Questions this analysis answers:**
- Which region generates the most revenue?
- What is the best acquisition channel?
- How are product prices distributed?
- Does offering higher discounts affect revenue?
- What are the strongest correlations between financial variables?

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100
print('Libraries loaded successfully')

## 3. Load Dataset

In [ ]:
df = pd.read_csv('../data/google_store_data.csv')

print(f'Shape: {df.shape}')
print(f'Total orders: {len(df):,}')
df.head()

## 4. Data Cleaning

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())

# Convert date columns
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Delivery Date'] = pd.to_datetime(df['Delivery Date'])

# Extract month and calculate delivery time
df['Month'] = df['Order Date'].dt.to_period('M').astype(str)
df['DeliveryDays'] = (df['Delivery Date'] - df['Order Date']).dt.days

print('\nData cleaned successfully.')

## 5. Exploratory Analysis

In [ ]:
print('=== Dataset Overview ===')
print(f'Total orders:    {len(df):,}')
print(f'Total revenue:   ${df["Revenue"].sum():,.2f}')
print(f'Average order:   ${df["Revenue"].mean():,.2f}')
print(f'Regions:         {df["Region"].nunique()}')
print(f'Categories:      {df["Product Category"].nunique()}')
print(f'Channels:        {list(df["Source"].unique())}')
print()
df[['Order Quantity', 'Unit Price', 'Revenue', 'Discount']].describe().round(2)

## 6. Data Visualizations

### 6.1 Total Revenue by Region

This chart shows which regions drive the most revenue, helping prioritize market focus.

In [ ]:
revenue_by_region = (
    df.groupby('Region')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

plt.figure(figsize=(10, 5))
sns.barplot(data=revenue_by_region, x='Region', y='Revenue', palette='Blues_d')
plt.title('Total Revenue by Region', fontsize=16, fontweight='bold')
plt.xlabel('Region', fontsize=12)
plt.ylabel('Total Revenue (USD)', fontsize=12)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../images/revenue_by_region.png', bbox_inches='tight')
plt.show()

### 6.2 Revenue by Acquisition Channel

Understanding which channel brings the most revenue helps guide marketing decisions.

In [ ]:
revenue_by_source = (
    df.groupby('Source')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

plt.figure(figsize=(10, 5))
sns.barplot(data=revenue_by_source, x='Source', y='Revenue', palette='muted')
plt.title('Total Revenue by Acquisition Channel', fontsize=16, fontweight='bold')
plt.xlabel('Channel', fontsize=12)
plt.ylabel('Total Revenue (USD)', fontsize=12)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../images/revenue_by_source.png', bbox_inches='tight')
plt.show()

### 6.3 Unit Price Distribution

A histogram with a density curve showing how product prices are spread.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['Unit Price'], bins=40, kde=True, color='steelblue')
plt.title('Unit Price Distribution', fontsize=16, fontweight='bold')
plt.xlabel('Unit Price (USD)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.tight_layout()
plt.savefig('../images/unit_price_distribution.png', bbox_inches='tight')
plt.show()

print(f'Median price: ${df["Unit Price"].median():.2f}')
print(f'Average price: ${df["Unit Price"].mean():.2f}')

### 6.4 Revenue by Customer Segment

Comparing revenue across customer segments to identify the most valuable audience.

In [ ]:
revenue_segment = (
    df.groupby('Customer Segment')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

plt.figure(figsize=(8, 5))
sns.barplot(data=revenue_segment, x='Customer Segment', y='Revenue', palette='Set2')
plt.title('Total Revenue by Customer Segment', fontsize=16, fontweight='bold')
plt.xlabel('Customer Segment', fontsize=12)
plt.ylabel('Total Revenue (USD)', fontsize=12)
plt.tight_layout()
plt.savefig('../images/revenue_by_segment.png', bbox_inches='tight')
plt.show()

### 6.5 Discount vs Revenue

Does giving more discounts lead to higher revenue? This regression plot shows the relationship.

In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(
    data=df, x='Discount', y='Revenue',
    scatter_kws={'alpha': 0.3, 'color': 'steelblue'},
    line_kws={'color': 'red'}
)
plt.title('Discount vs Revenue', fontsize=16, fontweight='bold')
plt.xlabel('Discount', fontsize=12)
plt.ylabel('Revenue (USD)', fontsize=12)
plt.tight_layout()
plt.savefig('../images/discount_vs_revenue.png', bbox_inches='tight')
plt.show()

### 6.6 Financial Correlation Heatmap

This heatmap shows how strongly the main financial variables correlate with each other.

In [ ]:
corr_cols = ['Order Quantity', 'Unit Price', 'Shipping Price', 'Total', 'Revenue', 'Discount']
corr = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='Blues', fmt='.2f', linewidths=0.5)
plt.title('Financial Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/correlation_heatmap.png', bbox_inches='tight')
plt.show()

## 7. Key Insights

- **Ontario** and **Atlantic** are the top revenue-generating regions.
- **Google Organic** and **Google Ads** are the strongest acquisition channels by total revenue.
- Most products are priced **below $50**, with a long tail of premium items above $100.
- **Discount and Revenue show a weak negative correlation** — heavy discounting does not drive higher revenue.
- **Total and Revenue** are strongly correlated (0.97+), confirming that order size is the main revenue driver.

## 8. Final Conclusion

This analysis of 18,782 Google Store orders reveals clear patterns in regional performance and customer behavior. Ontario and Atlantic lead in total revenue, while Google's own digital channels bring the highest-value customers.

The data shows that **discounting is not a reliable revenue lever** — high-revenue orders come from order size and unit price, not discount depth. Pricing strategy should focus on product mix and channel quality rather than blanket promotions.

---
*Analysis performed with Python · Pandas · NumPy · Matplotlib · Seaborn*